In [ ]:
# Setup: clone the StarX repo and the pinned TripoSR commit, install this
# notebook's dependencies, mount Drive, and report what machine we are on.
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "07"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR, TRIPOSR_DIR = "/content/StarX", "/content/TripoSR"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
    TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if not os.path.exists(TRIPOSR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git",
         TRIPOSR_DIR],
        check=True,
    )
subprocess.run(["git", "-C", TRIPOSR_DIR, "checkout", "-q", TRIPOSR_COMMIT], check=True)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync with starx/pins.py"
if pins.PIP_PINS[NOTEBOOK_ID]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )
# remove Colab preinstalls that break the pinned stack (see starx/pins.py)
if pins.PIP_UNINSTALL.get(NOTEBOOK_ID):
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-q", "-y",
         *pins.PIP_UNINSTALL[NOTEBOOK_ID]],
        check=False, capture_output=True,
    )

from starx import colab as scolab

DRIVE = scolab.mount_drive()
report = scolab.setup_report()

# 07 - Playground

Your sketches in, a 3D part out. This notebook loads the trained model and runs it on drawings you provide.

How to prepare drawings:

- dark strokes on light paper (a photo of a pencil drawing, a screenshot from any drawing app, or an exported CAD sketch);
- one image per sketch, uploaded in construction order - earlier images describe the base shape, later ones the features cut into it;
- the model was trained on top-down views of each sketch plane, so draw each profile flat, not in perspective.

No uploads handy? The notebook falls back to the repository's built-in demo design so every cell still runs. Use a GPU runtime; a reconstruction takes seconds.

In [ ]:
# Configuration - every tunable for this notebook lives here.
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from starx import checkpoint, rasterize, viz
from starx.config import StarXConfig, run_dir

RUN_NAME = "baseline_l4"   # the trained run to load
CKPT_STEP = None           # None: newest checkpoint
MC_RES = 256               # marching-cubes resolution for the export
N_ORBIT = 24               # frames in the turntable GIF
SEED = 1337

cfg = StarXConfig(
    drive_root=str(DRIVE / "StarX")
    if DRIVE is not None
    else os.path.join(REPO_DIR, "data", "StarX"),
    # surgery - must match the trained run
    max_sketch_channels=6,
    conv_init="i3d_mean",
    lora_r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    dino_layers=(0, 1, 2, 3),
    train_triplane_embed=False,
    sketch_size=512,
    bg_value=128,
    margin=0.05,
    mc_threshold=25.0,
    eval_chunk=131072,
    seed=SEED,
)
print(f"loading run {RUN_NAME}, up to {cfg.max_sketch_channels} sketch channels")

In [ ]:
# Build the model and load the trained checkpoint.
import torch

from starx import eval as seval
from starx import model as smodel

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _ = smodel.build_starx_model(cfg, TRIPOSR_DIR, device=device)
model.renderer.set_chunk_size(cfg.eval_chunk)
model.eval()

rdir = run_dir(cfg, RUN_NAME)
if CKPT_STEP is None:
    ckpt_path, ckpt_step = checkpoint.find_latest(rdir)
else:
    ckpt_step = CKPT_STEP
    ckpt_path = checkpoint.checkpoint_dir(rdir) / f"state_{CKPT_STEP:07d}.pt"
state = checkpoint.load_checkpoint(ckpt_path)
smodel.load_trainable_state_dict(model, state["model"])
print(f"loaded {RUN_NAME} at step {ckpt_step} on {device}")

In [ ]:
# Get drawings: upload PNGs on Colab (selection order = channel order), or
# fall back to the repo's reference design so the notebook always runs.
drawings = []
if IN_COLAB:
    from google.colab import files as colab_files

    print("upload your sketch images (or cancel to use the built-in demo)")
    try:
        uploaded = colab_files.upload()
    except Exception:
        uploaded = {}
    for name in uploaded:
        drawings.append(Image.open(io.BytesIO(uploaded[name])))
        print("loaded", name)

if not drawings:
    from starx import fusion

    fixture = fusion.load_design(
        Path(REPO_DIR) / "tests" / "fixtures" / "20203_7e31e92a_0000.json"
    )
    demo_stack, _ = rasterize.rasterize_design(fixture, cfg)
    drawings = [
        Image.fromarray(demo_stack[i])
        for i in range(len(fixture.sketches))
    ]
    print(f"no upload - using the demo design's {len(drawings)} sketches")

In [ ]:
# Turn the drawings into the model's input format: grayscale, darkened
# onto the training background, letterboxed to the canvas, stacked and
# padded to the channel count.
def prepare_channel(image, size, bg_value, margin):
    """One drawing (dark strokes on light paper) -> one uint8 channel."""
    gray = image.convert("L")
    target = int(size * (1.0 - 2.0 * margin))
    gray.thumbnail((target, target), Image.LANCZOS)
    # map paper white to the training background gray, keep strokes dark
    arr = (np.asarray(gray, dtype=np.float32) * (bg_value / 255.0)).astype(np.uint8)
    canvas = np.full((size, size), bg_value, dtype=np.uint8)
    top = (size - arr.shape[0]) // 2
    left = (size - arr.shape[1]) // 2
    canvas[top : top + arr.shape[0], left : left + arr.shape[1]] = arr
    return canvas


channels = [
    prepare_channel(img, cfg.sketch_size, cfg.bg_value, cfg.margin)
    for img in drawings[: cfg.max_sketch_channels]
]
while len(channels) < cfg.max_sketch_channels:
    channels.append(
        np.full((cfg.sketch_size, cfg.sketch_size), cfg.bg_value, dtype=np.uint8)
    )
play_stack = np.stack(channels)
play_tensor = rasterize.stack_to_tensor(play_stack)
fig = viz.show_sketch_stack(play_stack, title="your input, as the model sees it")
plt.show()

In [ ]:
# Reconstruct: sketch stack -> scene code -> rendered orbit.
with torch.no_grad():
    play_code = smodel.encode_sketches(model, play_tensor[None].to(device))[0]
    play_code = play_code.float()
orbit = model.render(
    play_code[None], n_views=8, elevation_deg=15.0,
    height=192, width=192, return_type="np",
)[0]
fig, axes = plt.subplots(1, 8, figsize=(15, 2.2))
for ax, view in zip(axes, orbit):
    ax.imshow(view)
    ax.axis("off")
fig.suptitle("the reconstruction from eight directions")
plt.show()

In [ ]:
# Extract the surface and inspect it interactively (drag to rotate).
play_mesh = seval.extract_mesh(model, play_code, res=MC_RES, threshold=cfg.mc_threshold)
assert play_mesh is not None, "no surface crossed the density threshold"
print(f"mesh: {len(play_mesh.vertices)} vertices, {len(play_mesh.faces)} faces")
fig = viz.mesh_side_by_side(None, play_mesh, title="your reconstruction")
fig.show()

In [ ]:
# A turntable GIF of your part.
from IPython.display import Image as IPImage
from IPython.display import display as ipy_display

frames = model.render(
    play_code[None], n_views=N_ORBIT, elevation_deg=15.0,
    height=256, width=256, return_type="np",
)[0]
gif_path = Path("/content/turntable.gif") if IN_COLAB else (
    Path(REPO_DIR) / "data" / "turntable.gif"
)
gif_path.parent.mkdir(parents=True, exist_ok=True)
viz.turntable_gif([(f * 255).astype(np.uint8) for f in frames], gif_path)
ipy_display(IPImage(data=gif_path.read_bytes(), format="gif"))

In [ ]:
# Export the mesh as an OBJ file you can open anywhere.
export_path = Path("/content/starx_reconstruction.obj") if IN_COLAB else (
    Path(REPO_DIR) / "data" / "starx_reconstruction.obj"
)
export_path.parent.mkdir(parents=True, exist_ok=True)
play_mesh.export(export_path)
print(f"saved {export_path} ({export_path.stat().st_size / 1024:.0f} KiB)")
if IN_COLAB:
    from google.colab import files as colab_files

    colab_files.download(str(export_path))

## Experiments to try

- Draw the same part twice with different stroke styles - how sensitive is the reconstruction?
- Shuffle the upload order: channel order encodes the construction sequence, so scrambling it should matter.
- Drop one sketch from a multi-sketch design and watch which feature disappears.
- Draw something the dataset never contains (a star-shaped plate, letters) and see how the model generalizes.
- Vary how large you draw within the page - the shared-scale convention means relative size carries meaning.